# Training Error, Testing Error & CIFAR-10

* Το CIFAR-10 είναι ένα υποσύνολο δεδομένων που αντιπροσοπεύει 60.000 εικόνες από 10 πιθανές κατηγορίες. Το βασικό πρόβλημα, λοιπόν, είναι αυτό της **ταξινόμησης** (classification). Θέλουμε να εκπαιδεύσουμε έναν αλγόριθμο, που μπορεί να "κοιτάξει" την εικόνα, και να μας πεί σε ποιά από τις 10 κατηγορίες ανήκει.

<img src="https://production-media.paperswithcode.com/datasets/4fdf2b82-2bc3-4f97-ba51-400322b228b1.png" width=550px/>

* Αυτό θεωρείται ένα από τα "απλά" προβλήματα της μηχανικής όρασης (computer vision). Παρόλα αυτά, βλέπουμε από τις παραπάνω εικόνες πως δεν είναι τετριμένο. Τα αντικείμενα που απεικονίζονται διαφέρουν αρκετά το ένα από το άλλο (ακόμα και αυτά που ανήκουν στην ίδια κατηγορία).

* Αυτο το πρόβλημα για μας είναι αφορμή να δούμε πως δουλεύουν στην πράξη διάφοροι αλγόριθμοι από την μηχανική μάθηση, και θα επιστρέψουμε σε αυτό το πρόβλημα στα επόμενα Notebook.

* Σε αυτό το Notebook, θα ξανακοιτάξουμε τα ρηχά και τα βαθιά δέντρα απόφασης, και επίσης την διαφορά μεταξύ του **Training Error** & **Testing Error** και έτσι είναι επίσης αφορμή να ξανακοιτάξουμε το φαινόμενο της **υπερμοντεολοποίησης** (overfitting).

```
Κωνσταντίνος Καραμανής: constantine@utexas.edu
http://users.ece.utexas.edu/~cmcaram/
The University of Texas at Austin
Archimedes/Athena RC
```

In [ ]:
# We import some of the basic packages we need.
import numpy as np
import matplotlib.pyplot as plt
from keras.datasets import cifar10
from sklearn import tree
from PIL import Image # powerful library for image processing tasks


# Ι. CIFAR-10 -- Μεταφορτώνοντας τα Δεδομένα

Στο πρώτο κομμάτι του Jupyter Notebook, θα μεταφορτώσουμε τα δεδομένα και θα τα επεξεργαστούμε.

Το CIFAR-10 είναι ένα σετ δεδομένων που περιέχει 60.000 είκόνες από δέκα κατηγορίες. Το κατεβάζουμε με την εντολή `load_data()`.  

Αυτή η εντολή μας επιτρέπει να διαχωρίσουμε τα δεδομένα σε δύο ομάδες, train και test. Η πρώτη ομάδα θα χρησιμοποιηθεί για να εκπαιδεύσουμε τους αλγορίθμους μας, και η δεύτερη για να εκτιμήσουμε την ακρίβεια.

In [ ]:
# μεταφορτώνουμε τα δεδομένα
(X_train, y_train), (X_test, y_test) = cifar10.load_data()


In [ ]:
# Έχουμε την δυνατότητα να ονομάσουμε εμείς τις 10 κατηγορίες
labels = ['αεροπλάνο', 'αυτοκίνητο', 'πτηνό', 'γάτα', 'ελάφι', 'σκύλος', 'βάτραχος', 'άλογο', 'πλοίο', 'φορτηγό']

# Πρώτος Βασικός Κανόνας: Κοιτάμε τα Δεδομένα

Μόλις κατεβάσαμε ένα σετ δεδομένων. Ο πρώτος βασικός κανόνας: κοιτάμε τα δεδομένα. Πόσα έχουμε; Τι σχήμα έχουνε;

Θα δούμε πως το σετ δεδομένων περιέχει 60.000 δείγματα. Από το συνολικό αυτόν αριθμό, τα 50.000 θα χρησιμοποιηθούν για εκπαίδευση (training), και τα υπόλοιπα 10.000 για την εκτίμηση ακρίβειας (testing).

In [ ]:
# Let's check the size
print('There are', X_train.shape[0], 'training points.')
print('There are', X_test.shape[0], 'testing points.')

In [ ]:
# How many times does each label appear?
for i in range(10): # a loop running from 0 to 9 (10 times in total)
  #ylist = y_train.tolist()
  num = sum(y_train==i) #calculates the sum of all the elements of y_train that are 0, 1 to 9 and stores it to variable num
  print('The label ', labels[i], 'appears ', num, 'times in the training data')#prints the number of times each label appears in y_train

Ας κοιτάξουμε πιο λεπτομερώς τα δεδομένα για να δούμε πώς αποθηκεύεται μία εικόνα με ψηφιακό τρόπο.

Βλέπουμε ότι τα χαρακτηριστικά (τι βρίσκεται στο X) αντιστοιχούν σε ένταση πίξελ σε RGB. Αυτό σημαίνει ότι κάθε εικόνα στο X αντιπροσωπεύεται ως 32 επί 32 πίξελ, με κάθε πίξελ να αντιστοιχεί σε τρεις αριθμούς. Αυτοί οι τρεις αριθμοί μεταξύ 0 και 255, δηλώνουν την ένταση τριών χρωμάτων, R για Κόκκινο (Red), G για Πράσινο (Green), και B για Μπλε (Blue), εξού και το RGB. Συνδυάζοντας αυτά τα τρία χρώματα μπορούμε να πάρουμε τα περισσότερα χρώματα του φάσματος. Ένα "μπλε" πίξελ του ουρανού, για παράδειγμα, θα είχε υψηλή ένταση για το μπλε, και πολύ χαμηλή (μέχρι και μηδενική) ένταση για το κόκκινο και το πράσινο. Ένα "λευκό" πίξελ θα είχε υψηλή ένταση και στα τρία χρώματα. Ένα "μαύρο" πίξελ θα είχε χαμηλή (ή ακόμα και μηδενική) ένταση σε όλα τα χρώματα. Για περισσότερες πληροφορίες σχετικά με τον τρόπο που αντιπροσωπεύονται ψηφιακά οι εικόνες, μπορείτε να διαβάσετε [εδώ](https://en.wikipedia.org/wiki/RGB_color_model).

In [ ]:
print('These are the labels', y_train)
print('The feature vectors have shape', X_train[0].shape)
print('This is what the feature vector actually looks like',X_train[0])

# Από 32x32x3 = 3.072 αριθμούς, Ταξινόμηση!

Τώρα βλέπουμε την πραγματική μορφή και διάσταση του προβλήματος: παρουσιάζουμε 32x32x3 = 3.072 αριθμούς στον αλγόριθμό μας, και με βάση αυτούς, πρέπει ο αλγόριθμος να απόφασίσει σε ποιά από τις 10 κατηγορίες ανήκει η φωτογραφία. Μπορεί να μας φαίνεται εύκολο όταν βλέπουμε τα δεδομένα σαν εικόνες, αλλά εάν το σκεφτούμε το πρόβλημα όπως το βλέπει ο αλγόριθμος φαίνεται μάλλον πιο δύσκολο. Δείτε τα παρακάτω δύο arrays -- το πρώτο αντιστοιχεί σε φορτηγό, και το δεύτερο σε ελάφι. Το μυαλό μας δύσκολα θα μπορούσε να βρεί κανόνες για να το καταλάβουμε αυτό.

In [ ]:
# NumPy's ravel() function flattens into a 1-d array
print('These are the first 1000 feature values of the 16th training point:',np.ravel(X_train[16])[0:1000])
print('\n\n These are the first 1000 feature values of the 904th training point:',np.ravel(X_train[904])[0:1000])


# Βλέπουμε τις Εικόνες

Αφού τα δεδομένα αντιπροσωπεύουν εικόνες, εμείς μπορούμε να ζητήσουμε από την Python να μας τις δείξει.

Ας αρχίσουμε με τις πάνω δύο εικόνες (φορτηγό και ελάφι).

In [ ]:
img = X_train[16] #assigns the ?th image from X_train to the variable img
plt.figure(figsize=(6, 3)) #creates a new figure of 6 inches width and 3 inches height
plt.imshow(img) #displays the image
plt.axis('off')  # Optional: turn off axes for a cleaner look
plt.show()


In [ ]:
img = X_train[904]
plt.figure(figsize=(6, 3))
plt.imshow(img)
plt.axis('off')
plt.show()

In [ ]:
# Με τον ίδιο τρόπο μπορούμε να δούμε πως μοιάζουν και άλλες εικόνες
fig, axes = plt.subplots(ncols=5, nrows=3, figsize=(17, 8)) #creates a new figure (fig) and a set of subplots (axes) arranged in a grid
#of 5 columns, 3 rows of 17 inches wide and 8 inches high images
#axes is an array of objects where each element corresponds to a subplot.
index = 0
for i in range(3):
    for j in range(5):
        axes[i,j].set_title(labels[y_train[index][0]])
        axes[i,j].imshow(X_train[index])
        axes[i,j].get_xaxis().set_visible(False)
        axes[i,j].get_yaxis().set_visible(False)
        index += 1
plt.show()


# Διαμόρφωση του Χ

Πολλοί αλγόριθμοι που θα χρησιμοποιήσουμε, όπως τα δέντρα απόφασης, θέλουνε τα δεδομένα με συγκεκριμένη μορφή: ο πίνακας $X$ έχει μία σειρά για κάθε δείγμα. Οπότε στην δική μας περίπτωση, κάθε εικόνα πρέπει να καταγράφεται σε μία σειρά του πίνακα. Αντί για 32 x 32 x 3, πρέπει να γίνει 1 x 3072. Αυτό το επιτυχαίνουμε με την εντολή `reshape`. Έτσι, τα δεδομένα μας που θα χρησιμοποιήσουμε για εκπαίδευση των αλγορίθμων, θα είναι αποθηκευμένα σε έναν πίνακα 50.000 x 3.072. Τα δεδομένα που θα χρησιμοποιήσουμε για να εκτιμήσουμε την αποτελεσματικότητα του αλγορίθμου, τα λεγόμενα Testing Data, αποθηκεύονται σε πίνακα 10.000 x 3.072.

In [ ]:
# We reformat X to have the required shape that our algorithms need.
X_tr = X_train.reshape((X_train.shape[0], -1))
print('This is now the shape of X:',X_tr.shape)
print('The reshaped X has ' + repr(X_tr.shape[0]) + ' rows, one for each sample, and ' + repr(X_tr.shape[1]) + ' columns, one for each feature.')#repr is used to obtain a string representation of an object

### Πολλά Είναι τα 50.000!

Μάλλον δεν χρειαζόμαστε 50.000 εικόνες για την εκπαίδευση.
Ο υπολογιστικός χρόνος για την εκπαίδευση θα μειωθεί τουλάχιστον 10Χ εάν χρησιμοποιήσουμε μόνο 5.000, και για αυτό το παράδειγμα μάλλον είναι αρκετά.

Φυσικά, μπορείτε να το δείτε και μόνοι σας αυτό: αλλάξτε τον αριθμό των δεδομένων που χρησιμοποιούμε στην εκπαίδευση, για να δείτε (α) την διαφορά στον υπολογιστικό χρόνο που απαιτείται στην εκπαίδευση, και (β) την βελτίωση της ακρίβειας στα δεδομένα εκτίμησης.

In [ ]:
# There are 50,000 data points. We pick a subset of them to speed up the training.
# The loader already shuffled, so it's safe to just pick the first ones.
N_tr = 5000
train_samples = N_tr
X_str = X_tr[:N_tr,:] # create a subset X_tr with N_tr samples.
y_str = y_train[:N_tr] # create  a subset target variable y_train with N_tr labels corresponding to the samples selected in X_str.
# check size
X_str.shape

### Υπολογίζουμε την ακρίβεια

Στο προηγούμενο Notebook υπολογίζαμε την ακρίβεια χρησιμοποιώντας μία μέθοδο από την sklearn.

Εδώ γράφουμε μόνοι μας την μέθοδο που υπολογίζει την ακρίβεια.

In [ ]:
# Let's write a simple function to compute the accuracy.

def compute_accuracy(array1, array2):
    if len(array1) != len(array2):
        raise ValueError("Both arrays must have the same number of elements")

    # Count the number of agreements
    agreement_count = sum(a == b for a, b in zip(array1, array2))#will give you the total number of positions where the corresponding elements in array1 and array2 are the same.

    # Calculate the fraction of agreement
    agreement_fraction = agreement_count / len(array1)

    return agreement_fraction

In [ ]:
# example of how it works
print(compute_accuracy([1,0,1,1,1,0,0,1,0,1],[0,1,1,1,1,0,0,1,1,0]))

# ΙΙ. Δέντρα Απόφασης

Τα Δέντρα Απόφασης (Decision Trees) είναι απλός αλλά θεμελιώδης αλγόριθμος στην μηχανική μάθηση. Προσπαθεί να βρεί απλούς κανόνες για να δημιουργήσει αποδοτικούς αλγορίθμους. Είχαμε κιόλας μία επαφή με τα δέντρα απόφασης στο Notebook οπού αναλύσαμε τα δεδομένα του Δαβήτη.

Τα δέντρα απόφασης χωρίζουν τα δεδομένα σε ομάδες, δημιουργώντας απλούς κανόνες της μορφής: $Χ_i > \alpha$.

Όπως έχουμε δεί, τα τρία βασικά βήματα είναι:

1. ```model = MyFamilyOfModels()``` -- εδώ ορίζουμε την οικογένεια μοντέλων.
2. ```model.fit(X_tr,y_tr)``` -- εδώ εκπαιδεύουμε το μοντέλο: βρίσκουμε το μοντέλο στην οικογένεια που ορίσαμε, που ελαχιστοποιεί τα λάθη, ή το training error, πάνω στα δεδομένα εκπαίδευσης (X_tr, y_tr).
3. ```model.predict(x)``` χρησιμοποιούμε το εκπαιδευμένο μοντέλο για να εκτιμήσουμε το ''y'' που πιστεύει το μοντέλο μας πως αντιστοιχεί στο x που του δώσαμε.

In [ ]:
# Ορίζουμε και εκπαιδεύουμε ένα δέντρο απόφασης: decision tree.
# Η μόνη παράμετρος που ορίζουμε είναι το βάθος: max_depth
decision_tree = tree.DecisionTreeClassifier(max_depth=1)


In [ ]:
# Το εκπαιδεύουμε:
decision_tree.fit(X_str, y_str)

# Έτσι μοιάζει το δέντρο μας:
tree.plot_tree(decision_tree, impurity = False)



In [ ]:
# Υπολογίζουμε την ακρίβεια στα δεδομένα εκπαίδευσης, τα training data

# Πρώτα χρησιμοποιούμε τον αλγόριθμό μας και σημειώνουμε τις εκτιμήσεις
# στα δεδομένα εκπαιδευσης.
# Μετα συγκρίνουμε τις προβλέψεις με τα πραγματικά labels.
yhat_train = decision_tree.predict(X_str)


print("Train score: %.4f" % compute_accuracy(y_str, yhat_train))

# Εναλακτικά μπορούμε να χρησιμοποιήσουμετ το model.score module.
train_score = decision_tree.score(X_str,y_str)
print("Train score: %.4f" % train_score)


# Πετύχαμε 0.1658: Αυτό είναι καλό;

Το 16.6% δεν μας γεμίζει το μάτι... αλλά ας μην ξεχνάμε πως έχουμε 10 πιθανές απαντήσεις, όχι μόνο 2 όπως είχαμε στο παράδειγμα του διαβήτη.

Οπότε, εάν μαντεύαμε τυχαία, θα περιμέναμε ακρίβεια πιο κοντά στο 10%, οπότε οτιδήποτε παραπάνω από 10% δείχνει πως κάποια τάση έχουμε πιάσει.

Ας το δούμε αυτό και μόνοι μας.

In [ ]:
# random guessing -- we are expecting something close to 0.1
import random

def generate_random_array(n):
    # Use a list comprehension to generate an array of n random integers between 0 and 9
    return [random.randint(0, 9) for _ in range(n)]

random_guess = generate_random_array(len(y_str))#the length of y_str is 5000
print("This is the accuracy of a random guess: %.4f" % compute_accuracy(y_str, random_guess))#prints accuracy to 4 decimal places


# Πιο Βαθιά Δέντρα Απόφασης

In [ ]:
# We now create a deeper decision tree.
decision_tree = tree.DecisionTreeClassifier(max_depth=4)

In [ ]:
# Now we actually train it -- this is the computationally intensive step
decision_tree.fit(X_str, y_str)

# This is what it looks like
tree.plot_tree(decision_tree)

# And this is its accuracy on the training data
train_score = decision_tree.score(X_str,y_str)
print("Train score: %.4f" % train_score)


# Ακρίβεια Εκπαίδευσης και Εκτίμησης

Το πιο βαθύ δέντρο τα πάει πολύ καλύτερα: φαίνεται πως έχει διπλασιάσει την ακρίβεια στα δεδομένα εκπαίδευσης.

Πως τα πάμε στα δεδομένα εκτίμησης;

Θα περιμέναμε πως η ακρίβεια στα δεδομένα εκτίμησης θα είναι **ψηλότερη** ή **χαμηλότερη** από την ακρίβεια στα δεδομένα εκπαίδευσης;


In [ ]:
# reshape the testing points like we did for training
X_t = X_test.reshape((X_test.shape[0], -1))
print(X_t.shape) #the 4D array becomes a 2D array 10000 by 3072

In [ ]:
yhat_test = decision_tree.predict(X_t)
print('This is the testing score: ' + repr(compute_accuracy(y_test, yhat_test)[0]) + '\nCompare this to the training score: ' + repr(train_score))

## Κάτι κάνουμε!

Φαίνεται πως τώρα έχουμε ακρίβεια 28% στα Training Data, και 23.6% στα Testing Data.

Το 23.6% είναι, όπως είπαμε, πολύ καλύτερο από το να μαντεύαμε τυχαία, μιά που έχουμε 10 κατηγορίες και τα δεδομένα είναι μοιρασαμένα και στις 10.

# Training vs Testing -- a closer look

Όπως κάναμε και με τα δεδομένα του διαβήτη, θα ξανακάνουμε το παραπάνω πείραμα, με δέντρα βάθους 1-15.

Μας ενδιαφέρει να συγκρίνουμε την ακρίβεια στα δύο σετ δεδομένων μας.

Μπορείτε να συγκρίνετε τον κώδικα να δείτε πως είναι πολύ παρόμοιος.

In [ ]:

training_scores = []
testing_scores = []
depth_values = range(15)
for depth in depth_values:
    dt = tree.DecisionTreeClassifier(max_depth=depth+1, criterion='gini')
    dt.fit(X_str,y_str)
    train_score = dt.score(X_str,y_str)
    test_score = dt.score(X_t, y_test)
    training_scores.append(train_score)
    testing_scores.append(test_score)

In [ ]:
print('These are the training scores:',training_scores)
print('These are the testing scores:',testing_scores)

In [ ]:
# Easier to see them on a graph, vs depth of tree
import matplotlib.pyplot as plt
fig, ax = plt.subplots()  # Create a figure containing a single axes.
ax.plot(training_scores,label='training')
ax.plot(testing_scores,label='testing')
ax.set_xlabel('tree depth')  # Add an x-label to the axes.
ax.set_ylabel('accuracy')  # Add a y-label to the axes.
ax.set_title("Train vs Test")  # Add a title to the axes.
ax.legend();  # Add a legend.

In [ ]:
# We can also visualize this as the difference in error between train and test errors

difference = []
zip_object = zip(training_scores, testing_scores)
for list1_i, list2_i in zip_object:
    difference.append(list1_i-list2_i)
fig2, ax2 = plt.subplots()  # Create a figure containing a single axes.
ax2.plot(difference,label='training')
ax.set_xlabel('tree depth')  # Add an x-label to the axes.
ax.set_ylabel('train - test')  # Add a y-label to the axes.
ax.set_title("Train minus Test")  # Add a title to the axes.
ax.legend();  # Add a legend.

# Training and Testing Error

Οπότε βλέπουμε πως η συμπεριφορά είναι πολύ παρόμοια με αυτήν που είδαμε στο παράδειγμα με τα δεδομένα του διαβήτη.

Και γενικότερα, τέτοια συμπεριφορά θα περιμέναμε με οποιονδήποτε κλασσικό αλγόριθμο της Μηχανικής Μάθησης.



## Μπορούμε να πετύχουμε καλύτερη ακρίβεια;

Αυτήν την ερώτηση την αφήνουμε για τα επόμενα Notebook, που να αναπτύξουμε ισχυρότερα εργαλεία (βαθιά νευρωνικά δίκτυα).